# Fitting a steady point-source with the public 14-year IceCube track data

This tutorial shows how to fit a neutrino point-like steady source (time-integrated) using the IceCube public 14-year point-source data with SkyLLH.
At the end of the notebook we also show how the same fit works using the older 10-year dataset.

We assume that you have installed the SkyLLH package in your environment by following the [Installation](../installation.rst) guide. If that's not the case you'll need to add the path to the SkyLLH folder to your PYTHONPATH.

In [ ]:
import numpy as np

## Initializing a configuration dictionary

First we have to create a config instance.

In [ ]:
from skyllh.core.config import Config

cfg = Config()

The Config object stores a dictionary with global settings, including computing, logging, and analysis settings. It is a required argument to several methods.

In [ ]:
cfg

### Multiprocessing

The multiprocessing in skyllh can be enabled globally by updating the configuration with a number of CPU cores.
It almost linearly speeds up creating an analysis and running trials, with a trade-off of a slightly higher memory usage:

In [ ]:
cfg.set_ncpu(6)

cfg

## Setting up logging

The user sets up the main logger. The logger name, logging format and level, output file, and other options can be set.

Default logging format and level are defined in the Config instance, but they can be overwritten by the arguments of `logging.setup_logging`.

In [ ]:
from skyllh.core.logging import setup_logging

logger = setup_logging(cfg=cfg, name='fitting_a_source', log_level='INFO')

## Getting the datasets

The datasets are automatically downloaded from [dataverse.harvard.edu](https://dataverse.harvard.edu/dataverse/icecube) to a local cache directory (`~/.skyllh/cache`) on first run. To use a custom dataset location, set `cfg['repository']['base_path']` to the desired path.

To create the correct dataset object, we need to import the 14-yr dataset definition:

In [ ]:
import skyllh

The collection of datasets is created with `skyllh.create_datasets`. With the default configuration, the data is downloaded automatically from `dataverse.harvard.edu`.

In [ ]:
help(skyllh.create_datasets)

If no `names` are passed to `skyllh.create_datasets`, it loads all data sets that are part of the IceTracks-DR2 dataset collection.

A tutorial on how to access specific information about which datasets are included in, e.g., IceTracks-DR2 is provided in [Dataset collections](dataset_collections.ipynb) tutorial.

In [ ]:
datasets = skyllh.create_datasets('IceTracks-DR2', cfg=cfg)

By default, this imports all available data sets in IceTracks-DR2 (all 14 years of data).

## Creating the analysis object

The analysis for doing a point-source search as in, e.g., https://doi.org/10.1103/PhysRevLett.124.051103 is pre-defined. This analysis instance can be created via the `create_analysis` method of the `time_integrated_ps` module of the public data interface.

The function requires the config, the datasets, and a source object. It also takes a number of additional optional arguments for which we refer the reader to the docstring.

In [ ]:
from skyllh.analyses.i3.publicdata_ps.time_integrated_ps import create_analysis

help(create_analysis)

Defining the Source: Here we use the location of NGC 1068 as an example source.

Instantiating a `PointLikeSource` object required the source location (R.A. and Dec.) in radians. Optionally, the source can have a `name` and a `weight`. The latter is in principle useful when performing a stacking analysis which is not supported for public data yet.

In [ ]:
from skyllh.core.source_model import PointLikeSource

help(PointLikeSource)

In [ ]:
src_ra = 40.67  # degrees
src_dec = -0.01  # degrees

source = PointLikeSource(ra=np.radians(src_ra), dec=np.radians(src_dec))

If not otherwise specified through the optional arguments, the `time_integrated_ps.create_analysis` method source assumes a signal hypothesis such that the defined source emits a powerlaw flux with spectral index $\gamma=2.0$:

$$\phi(E_{\nu}) = \mathrm{refplflux\_Phi0} \cdot \left(\frac{E_{\nu}}{\mathrm{refplflux\_E0}}\right)^{-\mathrm{refplflux\_gamma}}$$

In [ ]:
ana = create_analysis(cfg=cfg, datasets=datasets, source=source)

## Understanding how the analysis works

One usually wants to get best-fit values (maximum likelihood estimators) for the source parameters and a test-statistic, using methods like `ana.do_trial` or `ana.unblind` (see below). However, here we describe what happens under the hood when the analysis is run either on simulated or real experimental data.

### Initializing a trial

After the `Analysis` instance was created, the point-source analysis can be run. To do so the analysis is initialized with some _trial data_.
For instance we could initialize the analysis with the experimental data but the same applies for simulated data. 

The `Analysis` object provides the method `initialize_trial` to initialize a trial with data. It takes a list of `DataFieldRecordArray` instances holding the events. If we want to initialize a trial with the experimental data, we can get that list from the `Analysis` instance itself:

In [ ]:
events_list = [data.exp for data in ana.data_list]
ana.initialize_trial(events_list)

### Maximizing the log-likelihood ratio function

After initializing a trial, we can maximize the LLH ratio function using the `ana.llhratio.maximize` method. This method requires a ``RandomStateService`` instance in case the minimizer does not succeed and a new set of initial values for the fit parameters need to get generated. The method returns a 3-element tuple. The first element is the value of the LLH ratio function at its maximum. The second element is the set of fit parameters that maximize the LLH ratio. The third element is the status dictionary of the minimizer.

In [ ]:
from skyllh.core.random import RandomStateService

rss = RandomStateService(seed=1)

In [ ]:
(log_lambda_max, fitparam_values, status) = ana.llhratio.maximize(rss)

In [ ]:
print(f'log_lambda_max = {log_lambda_max}')
print(f'fitparam_values = {fitparam_values}')
print(f'status = {status}')

### Calculating the test-statistic

Using the maximum of the LLH ratio function and the fit parameter values at the maximum we can calculate the test-statistic using the `ana.calculate_test_statistic` method:

In [ ]:
TS = ana.calculate_test_statistic(log_lambda_max, fitparam_values)
print(f'TS = {TS:.3f}')

## Unblinding the data

Above, we perform the analysis following pedagogical steps: we initialize the analysis with a trial of the experimental data, maximize the log-likelihood ratio function for all given experimental data events, and calculate the test-statistic value.

However, the same calculations can be performed all together using the ``ana.unblind`` method of the analysis instance, which returns the test statistic and the best fit parameters directly.

Note that the ``ana.unblind`` method runs the analysis on the experimental data.

The methods to generate and analyze pseudo-data are illustrated in the `generating_pseudo_experiments.ipynb` notebook.

In [ ]:
help(ana.unblind)

In [ ]:
from skyllh.core.random import RandomStateService

rss = RandomStateService(seed=1)

ts, fitparam_values, status = ana.unblind(rss)

In [ ]:
print(f'TS = {ts:.3f}')
print(f'ns = {fitparam_values["ns"]:.2f}')
print(f'gamma = {fitparam_values["gamma"]:.2f}')
print(f'status = {status}')

## Converting the fit parameters into a flux normalization 

..todo: Tomas can you please a function that takes a specific flux param to get the conversion factor?

..todo: Further utility functions are available, 
1. ``ana.mu2flux(mu)`` returns a flux normalization at the chosen spectral index, chosen during the initialisation of the analysis instance. 
2. ``ana.flux2mu(flux_norm)`` returns the mean number of signal events for the given flux normalization value at the chosen spectral index.

By default the analysis assuming a powerlaw signal spectrum is created with a flux normalization of 1 $\text{GeV}^{-1}\text{s}^{-1}\text{cm}^{-2}\text{sr}^{-1}$ at 1 TeV (see `refplflux_Phi0` and `refplflux_E0` arguments of the `create_analysis` method).

The analysis instance offers the method `ana.calculate_fluxmodel_scaling_factor` that calculates the scaling factor that, for a given spectral model, converts a mean number of signal events of 1 into a flux normalization `refplflux_Phi0` at `refplflux_E0`.

In [ ]:
scaling_factor = ana.calculate_fluxmodel_scaling_factor()
print(f'1 event = {scaling_factor:.3e} (E/1000 GeV)^{{ -{fitparam_values["gamma"]:.2f} }} 1/(GeV s cm^2 sr)')

By default the conversion factor is returned for the spectral index with which the analysis

The method optionally takes a `fitparam_values` argument where a specific spectral index can be specified. Note that this won't change the internal `refplflux_gamma` value with which the analysis was initialized.

In [ ]:
scaling_factor = ana.calculate_fluxmodel_scaling_factor(fitparam_values=[1, 3])
print(f'1 event = {scaling_factor:.3e} (E/1000 GeV)^{{-3}} 1/(GeV s cm^2 sr)')

Alternatively, the `ana.mu2flux` and `ana.flux2mu` methods return

- a flux normalization at `refplflux_E0` given a certain mean number of signal events passed to it as an argument
- a mean number of signal events corresponding to a flux normalization at `refplflux_E0` passed to it as an argument

respectively.

They also take a different spectral parameters as optional arguments

In [ ]:
phi0 = ana.mu2flux(10)
print(f'Flux normalization for 10 events = {phi0:.3e} 1/(GeV s cm^2 sr)')

In [ ]:
mean_ns = ana.flux2mu(5e-14)
print(f'Mean number of events for a flux normalization at 1000 GeV of 5e-14 1/(GeV s cm^2 sr) = {mean_ns:.2f}')